## Alphabet Inc. (Google) (GOOG)  Exploratory Data Analysis

### Overview
This notebook presents an exploratory data analysis (EDA) of financial news headlines related to **Alphabet Inc. (Google) (GOOG)**. The analysis is part of a broader project at **Nova Financial Solutions** aimed at building a predictive analytics pipeline that connects market narratives to stock price movements.

### Objectives
- Understand the structure and statistical properties of the financial news dataset
- Identify common keywords, themes, and topics in GOOG-related headlines
- Analyze publication trends and volume spikes over time
- Examine publisher activity and contribution patterns

### Dataset
The analysis uses the **Financial News and Stock Price Integration Dataset (FNSPID)**, which contains financial news headlines, publication dates, publisher information, and associated stock ticker symbols.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import re
import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

print("✅ All libraries imported successfully!")

## 2. Loading the Data

We begin by loading the raw financial news dataset. The dataset contains news articles from multiple publishers with their publication dates and associated stock ticker symbols.

In [ ]:
df = pd.read_csv('../data/raw/raw_analyst_ratings.csv')

print(f"Dataset Shape: {df.shape}")
print(f"\nColumn Names: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

## 3. Data Cleaning & Filtering

Before analysis we clean the dataset by:
- Dropping the unnecessary unnamed index column
- Parsing the date column handling mixed timezone formats
- Fixing inconsistent publisher names
- Filtering for **GOOG** articles only

In [ ]:
df = df.drop(columns=['Unnamed: 0'])
df['date'] = pd.to_datetime(df['date'], format='mixed', utc=True)
df['publisher'] = df['publisher'].replace('Benzinga_Newsdesk', 'Benzinga Newsdesk')

goog_df = df[df['stock'] == 'GOOG'].copy().reset_index(drop=True)

print(f"Total GOOG articles: {len(goog_df)}")
print(f"\nDate range: {goog_df['date'].min().date()} to {goog_df['date'].max().date()}")
print(f"\nMissing values:\n{goog_df.isnull().sum()}")
goog_df.head()

## 4. Descriptive Statistics

In this section we examine the basic statistical properties of the dataset including headline length distribution, article count over time, and a general summary of the data.

In [ ]:
goog_df['headline_length'] = goog_df['headline'].apply(len)

print("=== Headline Length Statistics ===")
print(goog_df['headline_length'].describe())

print(f"\n=== Sample Headlines ===")
for h in goog_df['headline'].sample(5, random_state=42).values:
    print(f"• {h}")

### 4.1 Headline Length Distribution

We visualize the distribution of headline character lengths to understand how concise or verbose the financial news headlines tend to be.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(goog_df['headline_length'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Headline Length Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')
axes[0].axvline(goog_df['headline_length'].mean(), color='red', linestyle='--',
                label=f"Mean: {goog_df['headline_length'].mean():.1f}")
axes[0].legend()

axes[1].boxplot(goog_df['headline_length'], patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1].set_title('Headline Length Boxplot', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Character Count')

plt.suptitle('GOOG News Headline Length Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Publisher Analysis

We identify the most active publishers contributing to GOOG-related news. Understanding which sources dominate the coverage helps assess potential bias and the diversity of market perspectives.

In [ ]:
publisher_counts = goog_df['publisher'].value_counts()

print(f"Total unique publishers: {publisher_counts.nunique()}")
print(f"\nTop 10 most active publishers:")
print(publisher_counts.head(10))

### 5.1 Publisher Activity Visualization

The bar chart below highlights the dominance of certain publishers in GOOG news coverage, particularly **Benzinga Newsdesk** which accounts for the majority of articles.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

publisher_counts.head(10).plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')

ax.set_title('Top 10 Most Active Publishers — GOOG', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Articles')
ax.set_ylabel('Publisher')
ax.invert_yaxis()

for i, v in enumerate(publisher_counts.head(10)):
    ax.text(v + 0.5, i, str(v), va='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 5.2 Email Publisher Domain Analysis

Some publishers use email addresses as their names. We extract the domain from these emails to identify the organizations contributing to the news coverage.

In [ ]:
email_publishers = goog_df[goog_df['publisher'].str.contains('@', na=False)]

print(f"Articles from email-based publishers: {len(email_publishers)}")
print(f"Unique email publishers: {email_publishers['publisher'].nunique()}")

if len(email_publishers) > 0:
    email_publishers = email_publishers.copy()
    email_publishers['domain'] = email_publishers['publisher'].str.extract(r'@([\w.]+)')
    domain_counts = email_publishers['domain'].value_counts()
    print(f"\nTop domains:\n{domain_counts}")

    fig, ax = plt.subplots(figsize=(10, 5))
    domain_counts.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('Articles by Email Publisher Domain — GOOG',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Number of Articles')
    ax.set_ylabel('Domain')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("No email-based publishers found for this stock.")

## 6. Time Series Analysis of News Volume

We analyze how GOOG-related article publication frequency varies over time. Spikes in news volume often correlate with significant market events such as earnings releases, product launches, or macroeconomic developments.

In [ ]:
goog_df['date_only'] = goog_df['date'].dt.date
goog_df['hour'] = goog_df['date'].dt.hour
goog_df['day_of_week'] = goog_df['date'].dt.day_name()

daily_counts = goog_df.groupby('date_only').size()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].plot(daily_counts.index, daily_counts.values, color='steelblue', linewidth=1.5)
axes[0].fill_between(daily_counts.index, daily_counts.values, alpha=0.3, color='steelblue')
axes[0].set_title('Daily GOOG News Volume Over Time', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Number of Articles')

hour_counts = goog_df['hour'].value_counts().sort_index()
axes[1].bar(hour_counts.index, hour_counts.values, color='steelblue', edgecolor='white')
axes[1].set_title('Article Publication by Hour of Day', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Hour (UTC)')
axes[1].set_ylabel('Number of Articles')
axes[1].set_xticks(range(0, 24))

plt.suptitle('GOOG News Time Series Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Text Analysis & Topic Modeling

We apply NLP techniques to extract the most frequent keywords and themes from GOOG-related headlines. This helps identify recurring topics such as earnings, price targets, and product announcements that drive market attention. We use three complementary approaches: **TF-IDF**, **CountVectorizer**, and **LDA Topic Modeling**.

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_headline(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

goog_df['cleaned_headline'] = goog_df['headline'].apply(clean_headline)

# TF-IDF
tfidf = TfidfVectorizer(max_features=20, ngram_range=(1,2))
tfidf_matrix = tfidf.fit_transform(goog_df['cleaned_headline'])
tfidf_keywords = tfidf.get_feature_names_out()

# CountVectorizer
vectorizer = CountVectorizer(max_features=20, ngram_range=(1,2))
word_matrix = vectorizer.fit_transform(goog_df['cleaned_headline'])
word_freq = pd.DataFrame({
    'keyword': vectorizer.get_feature_names_out(),
    'frequency': word_matrix.toarray().sum(axis=0)
}).sort_values('frequency', ascending=False)

print("Top 20 Keywords (TF-IDF):")
print(tfidf_keywords)
print(f"\nTop 20 Keywords (Frequency):")
print(word_freq)

### 7.1 Top Keywords Visualization

The most frequently occurring keywords reveal the dominant themes in GOOG news coverage. Notable themes include YouTube and cloud computing growth, big tech sector competition with frequent mentions of Apple, Amazon and Facebook, and earnings performance — reflecting Alphabet's position as a key player in the broader technology sector during early 2020.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(word_freq['keyword'], word_freq['frequency'], color='steelblue', edgecolor='white')
axes[0].set_title('Top 20 Keywords by Frequency', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Frequency')
axes[0].set_ylabel('Keyword')
axes[0].invert_yaxis()

tfidf_scores = tfidf_matrix.toarray().mean(axis=0)
tfidf_df = pd.DataFrame({'keyword': tfidf_keywords, 'score': tfidf_scores})
tfidf_df = tfidf_df.sort_values('score', ascending=False)

axes[1].barh(tfidf_df['keyword'], tfidf_df['score'], color='coral', edgecolor='white')
axes[1].set_title('Top 20 Keywords by TF-IDF Score', fontsize=14, fontweight='bold')
axes[1].set_xlabel('TF-IDF Score')
axes[1].set_ylabel('Keyword')
axes[1].invert_yaxis()

plt.suptitle('GOOG News — Keyword & Topic Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 7.2 LDA Topic Modeling

Latent Dirichlet Allocation (LDA) is an unsupervised machine learning technique that discovers hidden topics in a collection of documents. We use LDA to identify the main themes discussed in GOOG-related headlines.

In [ ]:
# LDA Topic Modeling
lda_vectorizer = CountVectorizer(max_features=1000, ngram_range=(1,1))
lda_matrix = lda_vectorizer.fit_transform(goog_df['cleaned_headline'])

n_topics = 5
lda_model = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    max_iter=10
)
lda_model.fit(lda_matrix)

feature_names = lda_vectorizer.get_feature_names_out()

print("=== LDA Topic Modeling Results ===")
print(f"Number of topics: {n_topics}\n")

topics = []
for topic_idx, topic in enumerate(lda_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    topics.append(top_words)
    print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

# Visualize topics
fig, axes = plt.subplots(1, n_topics, figsize=(20, 4))

for topic_idx, topic in enumerate(lda_model.components_):
    top_indices = topic.argsort()[:-11:-1]
    top_words = [feature_names[i] for i in top_indices]
    top_scores = [topic[i] for i in top_indices]

    axes[topic_idx].barh(top_words, top_scores, color='steelblue', edgecolor='white')
    axes[topic_idx].set_title(f'Topic {topic_idx + 1}', fontsize=12, fontweight='bold')
    axes[topic_idx].invert_yaxis()
    axes[topic_idx].set_xlabel('Score')

plt.suptitle('GOOG — LDA Topic Modeling', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ LDA Topic Modeling complete!")

## 8. Key Findings & Summary

A summary of the most important insights derived from the exploratory data analysis of GOOG-related financial news headlines.

In [ ]:
print("=" * 55)
print("        GOOG EDA — KEY FINDINGS SUMMARY")
print("=" * 55)

print(f"""
📰 DATASET OVERVIEW
- Total GOOG articles analyzed : {len(goog_df)}
- Date range                   : {goog_df['date'].min().date()} to {goog_df['date'].max().date()}
- Unique publishers            : {goog_df['publisher'].nunique()}

📏 HEADLINE STATISTICS
- Average headline length      : {goog_df['headline_length'].mean():.1f} characters
- Shortest headline            : {goog_df['headline_length'].min()} characters
- Longest headline             : {goog_df['headline_length'].max()} characters

📰 PUBLISHER INSIGHTS
- Most active publisher        : {goog_df['publisher'].value_counts().index[0]}
- Articles by top publisher    : {goog_df['publisher'].value_counts().iloc[0]}

📈 NEWS VOLUME INSIGHTS
- Highest single day volume    : {daily_counts.max()} articles
- Date of highest volume       : {daily_counts.idxmax()}

🔑 TOP THEMES IDENTIFIED
- Price targets & analyst ratings
- COVID-19 / Coronavirus impact
- iPhone sales & product updates
- China operations & supply chain
- Stock trading & market movement
""")
print("=" * 55)